# Objective:  Verify that Code Exractor is functioning properly
## What does funtioning properly mean?

- **Identification** -> The LLM is properly identifying "Human-Percieved Code" (HPC)
  - **False Negative** -> HPC was missed from the original file
  - **False Positive** -> Objects that aren't considered HPC were extracted
- **Extraction** -> The LLM is properly extracting the HPC strings from the original files without altering them


---



### Test 1: Check Extraction

 - Search for extracted code blocks in the original file
  - Pass Criteria -> Every extracted string pairs directly with a string from the original file without repeats


---


### Test 2: Check Identification

Note: This is going to be inherently difficult because total verification of HPC requires a human. However there are some things that can be done to enhance results.

What is HPC?
- Actual code
- Object resembling code
  - What things make a string object resemble code?
    - Format
    - Python keywords [in coding context]
      - "If" in "If the dog barks" does not consistute a python IF keyword
      - "If-Statement" in "The If-Statment is complicated" is not actually HPC but instead an english noun not actually meant to resemble/be percieved as code.
    - Variable name references within english text (Ex: The value of 'x' is meant to grow exponentially)

Feasibility:
 - Actual code is verifiable with some formatting debugging
 - Code-like Format is potentially verifiable but difficult because GPT had inconsistent formatting
 - Variable name verification is continguent on identifying actual code and then looking for non-keywords within code that exist in other places

---

### Test 3: Potential LLM Tests involving recursive LLMs critiques
* Test A: LLM Critque that sees the full question and what was extracted and provides a GREEN or RED flag depending on its perception of extracted code

* Test B: LLM Critque to try and verify extracted human percieved code is valid, Hard code that removes valid extracted code out of original files, LLM code extract called again to look for residual human-percieved code (Loopable but contextually blind)

* Text C: LLM that extracts pure english from the questions enabling backwards verification

In [2]:
# SETUP BLOCK #
  # Mounts Drive
  # Import libraries
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import pandas as pd

Mounted at /content/drive


In [ ]:
# Directory Map

# Data
  # [a] questionTopic + model
        # Code_Blocks
            # setNum + question method + question topic + model + code_blocks.json
            # setNum + question method + question topic + model + code_blocks.json
            # ...
        # setNum + question method + question topic + model + .txt
        # setNum + question method + question topic + model + .txt
        # ...
  # [b] questionTopic + model
  # [c] questionTopic + model
  # ...

In [ ]:
import os
import json
import pandas as pd
import re
import unicodedata

# Normalize code lines: preserve structural meaning while collapsing only excessive internal whitespace
def normalize(s):
    s = unicodedata.normalize('NFKC', s)           # Normalize Unicode compatibility characters
    s = s.strip()
    s = s.replace('\t', '\\t').replace('\n', '\\n')  # Preserve visual formatting
    s = re.sub(r'\s+', ' ', s)                     # Collapse all whitespace (including Unicode) to single space
    return s

# Extract JSON block by scanning forward for first start delimiter and backward for last end delimiter
def extract_json_block(raw):
    # Possible start delimiters
    start_delims = ['```json', '"""json', "'''json"]
    end_delims = ['```', '"""', "'''"]

    start_idx = -1
    for delim in start_delims:
        idx = raw.find(delim)
        if idx != -1:
            start_idx = idx + len(delim)
            break

    if start_idx == -1:
        # Fallback: plain 'json' prefix
        if raw.startswith('json'):
            return raw[4:].strip()
        # Triple-quoted JSON fallback
        elif (raw.startswith('"""') and raw.endswith('"""')) or (raw.startswith("'''") and raw.endswith("'''")):
            return raw[3:-3].strip()
        else:
            return raw.strip()  # raw JSON with no wrapping

    # Find the **last** end delimiter after the start
    end_idx = -1
    for delim in end_delims:
        idx = raw.rfind(delim)
        if idx != -1 and idx > start_idx:
            end_idx = idx
            break

    if end_idx != -1:
        return raw[start_idx:end_idx].strip()
    else:
        return raw[start_idx:].strip()  # No clear end, go to end of file

# Define types of questions and model names used in the experiment
qTypes = ["If-Statements", "Loops"]
qModels = ["gpt-4o", "gpt-4o-mini", "o1", "o1-mini", "o3-mini"]

# Define DataFrame column structure (unused here, likely for future use)
df_columns = ['Filename', 'qType', 'qModel', 'qNumber', 'code_concatenated']
df_rows = []

# Initialize match and total counters
match_count = 0
total_count = 0

# Track performance stats by question type and model
type_model_stats = {
    qType: {
        qModel: {
            "match": 0,
            "total": 0,
            "parse_error": 0,
            "key": 0,
            "unicode": 0,
            "no_match": 0
        } for qModel in qModels
    } for qType in qTypes
}

# Iterate through each (qType, qModel) folder
for qType in qTypes:
    for qModel in qModels:
        Code_Blocks_Folder = f"/content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/{qType}_{qModel}/Code_Blocks"

        for code_filename in os.listdir(Code_Blocks_Folder):
            code_file_path = os.path.join(Code_Blocks_Folder, code_filename)
            if not os.path.isfile(code_file_path):
                continue

            try:
                with open(code_file_path) as f:
                    code_data = json.load(f)
            except json.JSONDecodeError:
                print(f"Invalid JSON Syntax in file: {code_filename}")
                type_model_stats[qType][qModel]["parse_error"] += 1
                continue

            for code_segment in code_data.get('code_segments', []):
                for code_line in code_segment.get('content', []):
                    total_count += 1
                    type_model_stats[qType][qModel]["total"] += 1

                    index_end = code_filename.find("_code_blocks")
                    original_file_name = code_filename[:index_end]
                    original_file_path = f"/content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/{qType}_{qModel}/{original_file_name}.txt"

                    try:
                        with open(original_file_path, 'r') as f:
                            try:
                                raw = f.read()
                                raw = extract_json_block(raw)
                                original_file_data = json.loads(raw)
                            except json.JSONDecodeError:
                                print(f"Could not parse JSON in file: {original_file_path}")
                                type_model_stats[qType][qModel]["parse_error"] += 1
                                continue

                            found = False
                            questions = original_file_data if isinstance(original_file_data, list) else original_file_data.get("questions", [])

                            try:
                                for question in questions:
                                    if question["difficulty"] == code_segment["difficulty"]:
                                        if normalize(code_line) in normalize(str(question)):
                                            found = True
                                            match_count += 1
                                            type_model_stats[qType][qModel]["match"] += 1
                                            break
                            except KeyError as e:
                                if not found:
                                    print(f"KeyError: {e} in file {original_file_path}")
                                    type_model_stats[qType][qModel]["key"] += 1
                                    continue

                            if found:
                                #print(f"PASS: '{repr(code_line)[1:-1]}' FOUND IN {original_file_path}\n")
                                continue
                            else:
                                print(f"Search Error: '{repr(code_line)[1:-1]}' NOT FOUND IN {original_file_path}\n")
                                type_model_stats[qType][qModel]["no_match"] += 1

                    except UnicodeDecodeError as u:
                        print(f"UnicodeDecodeError: {u} in file {original_file_path}")
                        type_model_stats[qType][qModel]["unicode"] += 1
                        continue

# ------------------- Final Summary -------------------
print(f"\n✅ Total Matches: {match_count} / {total_count}")
if total_count > 0:
    print(f"✅ Overall Match Rate: {match_count / total_count:.2%}")
else:
    print("⚠️ No code lines processed.")

# ------------------- Table Report -------------------
print("\n📊 Match & Error Breakdown by qType and qModel:")
header = f"{'qType':<15} | {'qModel':<10} | {'Match':>7} | {'Total':>7} | {'MatchRate':>10} | {'NoMatch':>8} | {'Parse':>5} | {'Key':>3} | {'Uni':>3}"
print(header)
print("-" * len(header))

for qType in qTypes:
    for qModel in qModels:
        stats = type_model_stats[qType][qModel]
        m, t = stats["match"], stats["total"]
        rate = (m / t * 100) if t > 0 else 0
        print(f"{qType:<15} | {qModel:<10} | {m:7d} | {t:7d} | {rate:9.2f}% | {stats['no_match']:8d} | {stats['parse_error']:5d} | {stats['key']:>3} | {stats['unicode']:>3}")


Streaming output truncated to the last 5000 lines.

PASS: 'list comprehension' FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/Loops_o1/set_9_multiple-choice_Loops_o1.txt

PASS: 'Python' FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/Loops_o1/set_9_multiple-choice_Loops_o1.txt

PASS: 'keyword' FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/Loops_o1/set_9_multiple-choice_Loops_o1.txt

PASS: 'Python' FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/Loops_o1/set_9_multiple-choice_Loops_o1.txt

PASS: 'list' FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/Loops_o1/set_9_multiple-choice_Loops_o1.txt

PASS: 'fruits' FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/Loops_o1/set_9_multiple-choice_Loops_o1.txt

PASS: 'for each fruit in fruits: print(fruit)' FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/Loops_o1/set_9_multiple-choice_Loops_o1.txt

PASS: 'for fruit in fru

In [ ]:
'''
✅ Total Matches: 15723 / 16538
✅ Overall Match Rate: 95.07%

📊 Match & Error Breakdown by qType and qModel:
qType           | qModel     |   Match |   Total |  MatchRate |  NoMatch | Parse | Key | Uni
--------------------------------------------------------------------------------------------
If-Statements   | gpt-4o     |    1530 |    1597 |     95.80% |        1 |     0 |  66 |   0
If-Statements   | gpt-4o-mini |    1387 |    1390 |     99.78% |        3 |     0 |   0 |   0
If-Statements   | o1         |    2637 |    2674 |     98.62% |       37 |     0 |   0 |   0
If-Statements   | o1-mini    |    1628 |    1927 |     84.48% |       12 |    94 | 193 |   0
If-Statements   | o3-mini    |    2070 |    2108 |     98.20% |       38 |     0 |   0 |   0
Loops           | gpt-4o     |    1179 |    1244 |     94.77% |        3 |     0 |  48 |  14
Loops           | gpt-4o-mini |    1240 |    1281 |     96.80% |       41 |     0 |   0 |   0
Loops           | o1         |    1759 |    1770 |     99.38% |       11 |     0 |   0 |   0
Loops           | o1-mini    |     988 |    1237 |     79.87% |        5 |    22 | 222 |   0
Loops           | o3-mini    |    1305 |    1310 |     99.62% |        5 |     0 |   0 |   0
'''

In [3]:
import os
import json
import pandas as pd
import re
import unicodedata

# Normalize code lines: preserve structural meaning while collapsing only excessive internal whitespace
def normalize(s):
    s = unicodedata.normalize('NFKC', s)           # Normalize Unicode compatibility characters
    s = s.strip()
    s = s.replace('\t', '\\t').replace('\n', '\\n')  # Preserve visual formatting
    s = re.sub(r'\s+', ' ', s)                     # Collapse all whitespace (including Unicode) to single space
    return s

# Extract JSON string by looking for the last closing delimiter (handles embedded delimiters better)
def extract_json_block(raw):
    start_delims = ['```json', '"""json', "'''json"]
    end_delims = ['```', '"""', "'''"]

    start_idx = -1
    for delim in start_delims:
        idx = raw.find(delim)
        if idx != -1:
            start_idx = idx + len(delim)
            break

    if start_idx == -1:
        if raw.startswith('json'):
            return raw[4:].strip()
        elif (raw.startswith('"""') and raw.endswith('"""')) or (raw.startswith("'''") and raw.endswith("'''")):
            return raw[3:-3].strip()
        else:
            return raw.strip()

    end_idx = -1
    for delim in end_delims:
        idx = raw.rfind(delim)
        if idx != -1 and idx > start_idx:
            end_idx = idx
            break

    return raw[start_idx:end_idx].strip() if end_idx != -1 else raw[start_idx:].strip()

# Define types of questions and model names used in the experiment
qTypes = ["If-Statements", "Loops"]
qModels = ["gpt-4o", "gpt-4o-mini", "o1", "o1-mini", "o3-mini"]

df_columns = ['Filename', 'qType', 'qModel', 'qNumber', 'code_concatenated']
df_rows = []

match_count = 0
total_count = 0

type_model_stats = {
    qType: {
        qModel: {
            "match": 0,
            "total": 0,
            "parse_error": 0,
            "key": 0,
            "unicode": 0,
            "no_match": 0
        } for qModel in qModels
    } for qType in qTypes
}

for qType in qTypes:
    for qModel in qModels:
        Code_Blocks_Folder = f"/content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/{qType}_{qModel}/Code_Blocks"

        for code_filename in os.listdir(Code_Blocks_Folder):
            code_file_path = os.path.join(Code_Blocks_Folder, code_filename)
            if not os.path.isfile(code_file_path):
                continue

            try:
                with open(code_file_path) as f:
                    code_data = json.load(f)
            except json.JSONDecodeError:
                print(f"Invalid JSON Syntax in file: {code_filename}")
                type_model_stats[qType][qModel]["parse_error"] += 1
                continue

            for i, code_segment in enumerate(code_data.get('code_segments', [])):
                for code_line in code_segment.get('content', []):
                    total_count += 1
                    type_model_stats[qType][qModel]["total"] += 1

                    index_end = code_filename.find("_code_blocks")
                    original_file_name = code_filename[:index_end]
                    original_file_path = f"/content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/{qType}_{qModel}/{original_file_name}.txt"

                    try:
                        with open(original_file_path, 'r') as f:
                            try:
                                raw = f.read().strip()
                                raw = extract_json_block(raw)
                                original_file_data = json.loads(raw)

                            except json.JSONDecodeError:
                                print(f"Could not parse JSON in file: {original_file_path}")
                                type_model_stats[qType][qModel]["parse_error"] += 1
                                continue

                            found = False
                            questions = original_file_data if isinstance(original_file_data, list) else original_file_data.get("questions", [])

                            try:
                                if len(questions) == len(code_data.get("code_segments", [])):
                                    # Match by order if counts match
                                    question = questions[i]
                                    if normalize(code_line) in normalize(str(question)):
                                        found = True
                                else:
                                    # Fallback: match by difficulty
                                    for question in questions:
                                        if question.get("difficulty") == code_segment.get("difficulty"):
                                            if normalize(code_line) in normalize(str(question)):
                                                found = True
                                                break

                                if found:
                                    match_count += 1
                                    type_model_stats[qType][qModel]["match"] += 1

                            except KeyError as e:
                                print(f"KeyError: {e} in file {original_file_path}")
                                type_model_stats[qType][qModel]["key"] += 1
                                continue

                            if found:
                                #print(f"PASS: '{repr(code_line)[1:-1]}' FOUND IN {original_file_path}\n")
                                continue
                            else:
                                print(f"Search Error: '{repr(code_line)[1:-1]}' NOT FOUND IN {original_file_path}\n")
                                type_model_stats[qType][qModel]["no_match"] += 1

                    except UnicodeDecodeError as u:
                        print(f"UnicodeDecodeError: {u} in file {original_file_path}")
                        type_model_stats[qType][qModel]["unicode"] += 1
                        continue

# ------------------- Final Summary -------------------
print(f"\n✅ Total Matches: {match_count} / {total_count}")
if total_count > 0:
    print(f"✅ Overall Match Rate: {match_count / total_count:.2%}")
else:
    print("⚠️ No code lines processed.")

# ------------------- Table Report -------------------
print("\n📊 Match & Error Breakdown by qType and qModel:")
header = f"{'qType':<15} | {'qModel':<10} | {'Match':>7} | {'Total':>7} | {'MatchRate':>10} | {'NoMatch':>8} | {'Parse':>5} | {'Key':>3} | {'Uni':>3}"
print(header)
print("-" * len(header))

for qType in qTypes:
    for qModel in qModels:
        stats = type_model_stats[qType][qModel]
        m, t = stats["match"], stats["total"]
        rate = (m / t * 100) if t > 0 else 0
        print(f"{qType:<15} | {qModel:<10} | {m:7d} | {t:7d} | {rate:9.2f}% | {stats['no_match']:8d} | {stats['parse_error']:5d} | {stats['key']:>3} | {stats['unicode']:>3}")


Search Error: '==' NOT FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/If-Statements_gpt-4o/set_1_true-false_If-Statements_gpt-4o.txt

Search Error: 'if x is greater than 10' NOT FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/If-Statements_gpt-4o-mini/set_4_drag-and-drop_If-Statements_gpt-4o-mini.txt

Search Error: 'if x is equal to 5' NOT FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/If-Statements_gpt-4o-mini/set_4_drag-and-drop_If-Statements_gpt-4o-mini.txt

Search Error: 'print('x is less than or equal to 10')' NOT FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/If-Statements_gpt-4o-mini/set_9_multiple-choice_If-Statements_gpt-4o-mini.txt

Search Error: 'fruit = 'apple'' NOT FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer25/Data/If-Statements_o1/set_10_drag-and-drop_If-Statements_o1.txt

Search Error: '[1] fruit [2] ['apple','banana']:' NOT FOUND IN /content/drive/Shareddrives/Lopez_Morrison_Summer

In [ ]:
"""
✅ Total Matches: 16139 / 16538
✅ Overall Match Rate: 97.59%

📊 Match & Error Breakdown by qType and qModel:
qType           | qModel     |   Match |   Total |  MatchRate |  NoMatch | Parse | Key | Uni
--------------------------------------------------------------------------------------------
If-Statements   | gpt-4o     |    1596 |    1597 |     99.94% |        1 |     0 |   0 |   0
If-Statements   | gpt-4o-mini |    1387 |    1390 |     99.78% |        3 |     0 |   0 |   0
If-Statements   | o1         |    2637 |    2674 |     98.62% |       37 |     0 |   0 |   0
If-Statements   | o1-mini    |    1820 |    1927 |     94.45% |       13 |    94 |   0 |   0
If-Statements   | o3-mini    |    2070 |    2108 |     98.20% |       38 |     0 |   0 |   0
Loops           | gpt-4o     |    1227 |    1244 |     98.63% |        3 |     0 |   0 |  14
Loops           | gpt-4o-mini |    1240 |    1281 |     96.80% |       41 |     0 |   0 |   0
Loops           | o1         |    1759 |    1770 |     99.38% |       11 |     0 |   0 |   0
Loops           | o1-mini    |    1098 |    1237 |     88.76% |      117 |    22 |   0 |   0
Loops           | o3-mini    |    1305 |    1310 |     99.62% |        5 |     0 |   0 |   0
"""